In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Ord, Alias

# ================================================================
# 1) CARGA DE DATOS Y CÁLCULO DE V_MAX
# ================================================================
BASE_DIR = Path('.') 

# Lectura de archivos reales
ret_spx = pd.read_csv(BASE_DIR / "ret_semanal_spx.csv")
ret_cmc = pd.read_csv(BASE_DIR / "ret_semanal_cmc200.csv")
prob_spx = pd.read_csv(BASE_DIR / "prob_spx.csv").melt(id_vars='t', var_name='k', value_name='value')
prob_cmc = pd.read_csv(BASE_DIR / "prob_cmc200.csv").melt(id_vars='t', var_name='k', value_name='value')

# Consolidación para cálculo de V_max y Wealth Index
r_data_raw = pd.merge(ret_spx, ret_cmc, on='t').drop_duplicates().set_index('t')
r_data_raw.columns = ['SPX', 'CMC200']

# Definición de V_max: Varianza del activo estable + 20% de holgura
V_max = r_data_raw['SPX'].var() * 1.2
print(f"Presupuesto de Riesgo (V_max) calculado: {V_max:.8f}")

# ================================================================
# 2) MODELO DE OPTIMIZACIÓN (GAMSPy)
# ================================================================
m = Container()

# Sets
t_sorted = sorted(ret_spx['t'].unique(), key=lambda x: int(x[1:]))
t = Set(m, name="t", records=t_sorted)
i = Set(m, name="i", records=["SPX", "CMC200"])
j = Alias(m, name="j", alias_with=i)
k = Set(m, name="k", records=["bear", "bull"])

# Parámetros de Retornos
r = Parameter(m, name="r", domain=[i, t])
r_recs = pd.concat([
    ret_spx.assign(i="SPX").rename(columns={"ret_semanal_spx": "value"}),
    ret_cmc.assign(i="CMC200").rename(columns={"ret_semanal_cmc200": "value"})
])
r.setRecords(r_recs[['i', 't', 'value']])

# Parámetros de Probabilidades
p = Parameter(m, name="p", domain=[i, k, t])
p_recs = pd.concat([prob_spx.assign(i="SPX"), prob_cmc.assign(i="CMC200")])
p.setRecords(p_recs[['i', 'k', 't', 'value']])

# Parámetros de Control
lambda_riesgo = Parameter(m, name="lambda_riesgo")
c_base_recs = pd.DataFrame([["SPX", 0.001], ["CMC200", 0.004]], columns=['i', 'value'])
c_base = Parameter(m, name="c_base", domain=[i], records=c_base_recs)
c_mult = Parameter(m, name="c_mult")
w0 = Parameter(m, name="w0", domain=[i], records=pd.DataFrame([["SPX", 0.5], ["CMC200", 0.5]], columns=['i', 'value']))

# --- Cálculo de Momentos (Mezcla de distribuciones) ---
den_mu = Parameter(m, domain=[i, k]); den_mu[i, k] = Sum(t, p[i, k, t])
mu_hat = Parameter(m, domain=[i, k])
mu_hat[i, k].where[den_mu[i, k] > 0] = Sum(t, p[i, k, t] * r[i, t]) / den_mu[i, k]
mu_mix = Parameter(m, name="mu_mix", domain=[i, t])
mu_mix[i, t] = Sum(k, p[i, k, t] * mu_hat[i, k])

den_sig = Parameter(m, domain=[i, j, k]); den_sig[i, j, k] = Sum(t, p[i, k, t] * p[j, k, t])
sigma_hat = Parameter(m, domain=[i, j, k])
sigma_hat[i, j, k].where[den_sig[i, j, k] > 0] = Sum(t, p[i, k, t] * p[j, k, t] * (r[i, t] - mu_hat[i, k]) * (r[j, t] - mu_hat[j, k])) / den_sig[i, j, k]
sigma_tmp = Parameter(m, domain=[i, j, t]); sigma_tmp[i, j, t] = Sum(k, p[i, k, t] * p[j, k, t] * sigma_hat[i, j, k])
sigma_mix = Parameter(m, name="sigma_mix", domain=[i, j, t])
sigma_mix[i, j, t] = 0.5 * (sigma_tmp[i, j, t] + sigma_tmp[j, i, t])
sigma_mix[i, j, t].where[i.sameAs(j)] = sigma_mix[i, j, t] + 0.000001

# --- Ecuaciones del Modelo ---
w = Variable(m, name="w", domain=[i, t], type="Positive")
u = Variable(m, name="u", domain=[i, t], type="Positive")
v = Variable(m, name="v", domain=[i, t], type="Positive")
z = Variable(m, name="z")

fo = Equation(m, name="fo")
# FO: Max Retorno - Lambda*(Riesgo - V_max) - Costos(u+v)
fo[...] = z == Sum(t, 
    Sum(i, w[i, t] * mu_mix[i, t]) - 
    lambda_riesgo * (Sum((i, j), w[i, t] * w[j, t] * sigma_mix[i, j, t]) - V_max) -
    Sum(i, c_base[i] * c_mult * (u[i, t] + v[i, t]))
)

eq_budget = Equation(m, name="eq_budget", domain=[t])
eq_budget[t] = Sum(i, w[i, t]) == 1.0

eq_rebalance = Equation(m, name="eq_rebalance", domain=[i, t])
eq_rebalance[i, t].where[Ord(t) > 1] = (w[i, t] - w[i, t.lag(1)]) == (u[i, t] - v[i, t])

eq_anchor = Equation(m, name="eq_anchor", domain=[i, t])
eq_anchor[i, t].where[Ord(t) == 1] = (w[i, t] - w0[i]) == (u[i, t] - v[i, t])

portfolio_model = Model(m, name="Portafolio", equations=m.getEquations(), problem="QCP", sense=Sense.MAX, objective=z)

# ================================================================
# 3) EJECUCIÓN DE SENSIBILIDAD (VALORES SOLICITADOS)
# ================================================================
resultados_lista = []

print("Ejecutando escenarios de sensibilidad...")
for l_val in [0.3, 0.7, 1.1]:
    for c_val in [0.01, 0.2, 0.5]:
        lambda_riesgo.setRecords(l_val)
        c_mult.setRecords(c_val)
        portfolio_model.solve(output=None)
        
        df_temp = w.records.copy()[['t', 'i', 'level']]
        df_temp['Lambda'] = l_val
        df_temp['Costo_Mult'] = c_val
        resultados_lista.append(df_temp)

df_final = pd.concat(resultados_lista).rename(columns={'level': 'peso'})

# ================================================================
# 4) CÁLCULO DE WEALTH INDEX Y BENCHMARKS
# ================================================================
def calcular_wealth_index(df_pesos_esc, r_raw, c_info, m_c):
    piv = df_pesos_esc.pivot(index='t', columns='i', values='peso').loc[t_sorted].fillna(0)
    cambios = piv.diff().fillna(0).abs()
    # Costos operativos: (sum de delta_w * c_base) * c_mult
    costos = (cambios['SPX']*0.001 + cambios['CMC200']*0.004) * m_c
    ret_neto = (piv * r_raw).sum(axis=1) - costos
    return (1 + ret_neto).cumprod()

# Benchmark 1/N: Rebalanceo semanal al 50/50
wealth_1n = (1 + (0.5 * r_data_raw['SPX'] + 0.5 * r_data_raw['CMC200'])).cumprod()

# Benchmark Buy & Hold: Inversión inicial 50/50 estática
wealth_bh = (0.5 * (1 + r_data_raw['SPX']).cumprod() + 0.5 * (1 + r_data_raw['CMC200']).cumprod())

# ================================================================
# 5) REPORTES GRÁFICOS
# ================================================================

# --- Gráfico 1: Evolución de Capitales ---
plt.figure(figsize=(14, 7))
colors = plt.cm.plasma(np.linspace(0, 1, 9))

for idx, ((l, c), group) in enumerate(df_final.groupby(['Lambda', 'Costo_Mult'])):
    w_index = calcular_wealth_index(group, r_data_raw, c_base_recs, c)
    plt.plot(t_sorted, w_index, label=f'Opt L={l}, C={c}', color=colors[idx], alpha=0.6)

plt.plot(t_sorted, wealth_1n, label='BENCHMARK: 1/N (Rebal)', color='black', linestyle='--', linewidth=2)
plt.plot(t_sorted, wealth_bh, label='BENCHMARK: Buy & Hold 50/50', color='red', linestyle=':', linewidth=2)

plt.title(f'Wealth Index vs Benchmarks (Presupuesto Riesgo V = {V_max:.6f})')
plt.xticks(t_sorted[::15], rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Gráfico 2: Evolución de Pesos (Grilla 3x3) ---
lambdas = [0.3, 0.7, 1.1]
costos = [0.01, 0.2, 0.5]
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True, sharey=True)
fig.suptitle(f'Evolución de Pesos (SPX vs CMC200) | Restricción V = {V_max:.6f}', fontsize=16)

for i_idx, l_val in enumerate(lambdas):
    for j_idx, c_val in enumerate(costos):
        ax = axes[i_idx, j_idx]
        subset = df_final[(df_final['Lambda'] == l_val) & (df_final['Costo_Mult'] == c_val)]
        piv = subset.pivot(index='t', columns='i', values='peso').loc[t_sorted]
        ax.stackplot(t_sorted, piv['SPX'], piv['CMC200'], labels=['SPX', 'CMC200'], colors=['#1f77b4', '#ff7f0e'])
        ax.set_title(f'L: {l_val} | C: {c_val}')
        ax.set_ylim(0, 1)

plt.xticks(t_sorted[::20], rotation=45)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

Presupuesto de Riesgo (V_max) calculado: 0.00064596


RuntimeError: Could not load shared library C:\Users\Alumn. Prof. JPerez\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gamspy_base\gmdcclib64.dll: Windows error